In [1]:
import json

# ============================================================
# CONFIG
# ============================================================

# FILE_PATH = r"G:\V5-dataset\V5C\V5C.jsonl"

FILE_PATH = r"G:\V5-dataset\V5C\V5C.jsonl"  # Change this path to your dataset file

# ============================================================
# SUMMARY SIZE CLASSIFIER
# ============================================================

def classify_summary(summary):
    if not isinstance(summary, str) or not summary.strip():
        return "NO SUMMARY"

    length = len(summary.strip())

    if length <= 150:
        return "SMALL"
    elif length <= 250:
        return "MEDIUM"
    else:
        return "LARGE"


# ============================================================
# CHECK DATASET
# ============================================================

small = []
medium = []
large = []
no_summary = []

total = 0
invalid_json = 0

with open(FILE_PATH, "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, 1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            invalid_json += 1
            continue

        total += 1

        summary = (
            record
            .get("output", {})
            .get("summary", "")
        )

        size = classify_summary(summary)

        item = {
            "line": line_number,
            "length": len(summary.strip()) if isinstance(summary, str) else 0,
            "summary": summary
        }

        if size == "SMALL":
            small.append(item)

        elif size == "MEDIUM":
            medium.append(item)

        elif size == "LARGE":
            large.append(item)

        else:
            no_summary.append(item)


# ============================================================
# PRINT RESULTS
# ============================================================

print("=" * 80)
print("V5C SUMMARY SIZE CHECK")
print("=" * 80)

print(f"Total records     : {total}")
print(f"Invalid JSON      : {invalid_json}")
print()

print(f"SMALL             : {len(small):,}")
print(f"MEDIUM            : {len(medium):,}")
print(f"LARGE             : {len(large):,}")
print(f"NO SUMMARY        : {len(no_summary):,}")

print("=" * 80)

# ============================================================
# PERCENTAGES
# ============================================================

if total > 0:

    print("\nPERCENTAGE DISTRIBUTION")
    print("-" * 80)

    print(f"SMALL      : {len(small):,} "
          f"({len(small) / total * 100:.2f}%)")

    print(f"MEDIUM     : {len(medium):,} "
          f"({len(medium) / total * 100:.2f}%)")

    print(f"LARGE      : {len(large):,} "
          f"({len(large) / total * 100:.2f}%)")

    print(f"NO SUMMARY : {len(no_summary):,} "
          f"({len(no_summary) / total * 100:.2f}%)")


# ============================================================
# LENGTH STATISTICS
# ============================================================

all_lengths = [
    item["length"]
    for item in small + medium + large
]

if all_lengths:

    print("\nSUMMARY LENGTH STATISTICS")
    print("-" * 80)

    print(f"Minimum length : {min(all_lengths)} characters")
    print(f"Maximum length : {max(all_lengths)} characters")
    print(f"Average length : {sum(all_lengths) / len(all_lengths):.2f} characters")


# ============================================================
# SHOW NO-SUMMARY RECORDS
# ============================================================

if no_summary:

    print("\nNO SUMMARY RECORDS")
    print("-" * 80)

    for item in no_summary[:20]:
        print(
            f"Line {item['line']} | "
            f"Length: {item['length']}"
        )


# ============================================================
# SHOW LARGEST SUMMARIES
# ============================================================

print("\nTOP 10 LARGEST SUMMARIES")
print("-" * 80)

for item in sorted(
    large,
    key=lambda x: x["length"],
    reverse=True
)[:10]:

    print(
        f"Line {item['line']} | "
        f"{item['length']} characters | "
        f"{item['summary']}"
    )


# ============================================================
# SHOW SMALLEST SUMMARIES
# ============================================================

print("\nTOP 10 SMALLEST SUMMARIES")
print("-" * 80)

for item in sorted(
    small,
    key=lambda x: x["length"]
)[:10]:

    print(
        f"Line {item['line']} | "
        f"{item['length']} characters | "
        f"{item['summary']}"
    )

V5C SUMMARY SIZE CHECK
Total records     : 38039
Invalid JSON      : 0

SMALL             : 935
MEDIUM            : 27,416
LARGE             : 9,681
NO SUMMARY        : 7

PERCENTAGE DISTRIBUTION
--------------------------------------------------------------------------------
SMALL      : 935 (2.46%)
MEDIUM     : 27,416 (72.07%)
LARGE      : 9,681 (25.45%)
NO SUMMARY : 7 (0.02%)

SUMMARY LENGTH STATISTICS
--------------------------------------------------------------------------------
Minimum length : 90 characters
Maximum length : 447 characters
Average length : 225.83 characters

NO SUMMARY RECORDS
--------------------------------------------------------------------------------
Line 37688 | Length: 0
Line 37689 | Length: 0
Line 37690 | Length: 0
Line 37691 | Length: 0
Line 37692 | Length: 0
Line 37831 | Length: 0
Line 37832 | Length: 0

TOP 10 LARGEST SUMMARIES
--------------------------------------------------------------------------------
Line 16865 | 447 characters | The user cons

In [4]:
import json

# ============================================================
# CONFIG
# ============================================================

INPUT_PATH = r"G:\V5-dataset\V5C\V5C.jsonl"
OUTPUT_PATH = r"G:\V5-dataset\V5C\V5C_no_summary.jsonl"


# ============================================================
# EXTRACT NO-SUMMARY RECORDS
# ============================================================

total = 0
no_summary = 0
invalid_json = 0

with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as outfile:

    for line_number, line in enumerate(infile, 1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            invalid_json += 1
            continue

        total += 1

        # Safely get summary
        output = record.get("output", {})
        summary = output.get("summary", "") if isinstance(output, dict) else ""

        # Check for missing / null / blank summary
        if (
            summary is None
            or not isinstance(summary, str)
            or not summary.strip()
        ):
            outfile.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )
            no_summary += 1


# ============================================================
# RESULT
# ============================================================

print("=" * 70)
print("V5C NO-SUMMARY EXTRACTION")
print("=" * 70)

print(f"Total records       : {total:,}")
print(f"Invalid JSON        : {invalid_json:,}")
print(f"No-summary records  : {no_summary:,}")
print()

print(f"Output file:")
print(OUTPUT_PATH)

print("=" * 70)

if no_summary == 0:
    print("✓ NO EMPTY/MISSING SUMMARIES FOUND")
else:
    print(f"✓ Extracted {no_summary:,} records")

V5C NO-SUMMARY EXTRACTION
Total records       : 38,039
Invalid JSON        : 0
No-summary records  : 7

Output file:
G:\V5-dataset\V5C\V5C_no_summary.jsonl
✓ Extracted 7 records
